# Patient list with checksums (M1-T03)

Reads every archive in `MyDrive/BrainMRI_Data/raw/` **without extracting it** and builds:

| File (saved to `MyDrive/BrainMRI_Data/manifests/`) | One row per | Contents |
| --- | --- | --- |
| `<cohort>_manifest.csv` | file | patient, timepoint, scan type, size, SHA-256, image shape and voxel size; label values for masks |
| `unified_subject_index.csv` | patient + timepoint | which scans exist, their checksums, shape, label values, quality flags (columns from `docs/templates/DATASET_MANIFEST_TEMPLATE.csv`) |

It also checks each dataset against its expected size and, for RHUH-GBM, against TCIA's own checksum file.

**How to run:** CPU runtime is fine. **Runtime → Run all.** Reading about 55 GB from Drive takes roughly 30–60 minutes.
Datasets already listed are skipped on the next run, so after downloading BraTS-Africa and UTSW just run it again.

**When it finishes:** send the output of step 4 (the summary) to the team, then copy the CSVs into the repo's `data/manifests/`
and commit them on the `M1-T03-patient-list` branch.

In [ ]:
# 1. Settings, and connect Drive
REBUILD = False     # True = list every dataset again, even if its manifest already exists

import os, re, io, gzip, glob, tarfile, hashlib, datetime, collections
import numpy as np, pandas as pd
!pip -q install nibabel
import nibabel as nib
from google.colab import drive
drive.mount("/content/drive")

RAW = "/content/drive/MyDrive/BrainMRI_Data/raw"
OUT = "/content/drive/MyDrive/BrainMRI_Data/manifests"
os.makedirs(OUT, exist_ok=True)

# cohort key -> Drive folder, how to recognise patient and timepoint in the file paths, and facts from docs/dataset_research.md
COHORTS = {
    "brats2021": dict(folder="brats2021", single_timepoint=True, subject=r"(BraTS2021_\d{5})", timepoint=None,
                      expect=(1251, 1251), label_scheme="brats2021_1_2_4", et=4, rc=None,
                      stage="pre_treatment", seg_type="manual_expert_approved", refinement="manual",
                      prep="skull_stripped_sri24_1mm", order="not_available", tumor="glioma", site="", country="",
                      license="CC BY 4.0 (BraTS 2021, TCIA DOI 10.7937/jc8x-9874; Kaggle mirror dschettler8845/brats-2021-task1)"),
    "rhuh": dict(folder="longitudinal_rhuh_gbm", single_timepoint=False, subject=r"(RHUH-\d{4})", timepoint=None,
                 expect=(40, 120), label_scheme="rhuh_necrosis_peritumoral_enhancing", et=None, rc=None,
                 stage="", seg_type="manual_expert", refinement="manual",
                 prep="skull_stripped_sri24_1mm_zscored", order="timepoint_index", tumor="glioblastoma",
                 site="Rio Hortega University Hospital", country="Spain",
                 license="CC BY 4.0 (TCIA RHUH-GBM, DOI 10.7937/4545-c905)"),
    "mu": dict(folder="longitudinal_mu_glioma_post", single_timepoint=False, subject=r"(PatientID_\d+)", timepoint=r"(Timepoint_\d+)",
               expect=(203, 596), label_scheme="brats2024_1_2_3_4", et=3, rc=4,
               stage="post_treatment", seg_type="auto_then_neuroradiologist_refined", refinement="auto_refined",
               prep="skull_stripped_sri24_1mm_n4", order="relative_days", tumor="glioma",
               site="University of Missouri", country="USA",
               license="CC BY 4.0 (TCIA MU-Glioma-Post, DOI 10.7937/7k9k-3c83)"),
    "africa": dict(folder="external_brats_africa", single_timepoint=False, subject=r"(BraTS-[A-Za-z]+-\d{5})", timepoint=r"BraTS-[A-Za-z]+-\d{5}-(\d{3})",
                   expect=(146, 146), label_scheme="brats2023_1_2_3", et=3, rc=None,
                   stage="pre_treatment", seg_type="semi_automatic_radiologist_refined", refinement="auto_refined",
                   prep="skull_stripped_sri24_1mm", order="not_available", tumor="",
                   site="Nigerian centres (BraTS-Africa)", country="Nigeria",
                   license="CC BY 4.0 (TCIA BraTS-Africa, DOI 10.7937/v8h6-8x67)"),
    "utsw": dict(folder="external_utsw_glioma", single_timepoint=True, subject=None, timepoint=None,
                 expect=(625, 625), label_scheme="fets_1_2_4_to_verify", et=None, rc=None,
                 stage="pre_treatment", seg_type="", refinement="",
                 prep="skull_stripped_sri24_1mm", order="not_available", tumor="glioma",
                 site="UT Southwestern Medical Center", country="USA",
                 license="CC BY 4.0 (TCIA UTSW-Glioma, DOI 10.7937/dfae-1b86)"),
}
present = {k: v for k, v in COHORTS.items() if glob.glob(f"{RAW}/{v['folder']}/*.tar")}
print("Datasets found on Drive:", list(present) or "none")
print("Not downloaded yet:     ", [k for k in COHORTS if k not in present] or "none")

In [ ]:
# 2. How file names are read: scan type, patient and timepoint
def scan_type(filename):
    name = re.sub(r"\.nii(\.gz)?$", "", filename.lower())
    tokens = set(re.split(r"[_\-\.\s]+", name))
    if tokens & {"seg", "segmentation", "segmentations", "mask", "tumormask", "label", "labels", "labelmap"} \
            or "seg" in name or "mask" in name:
        return "seg"
    if "adc" in tokens:
        return "adc"
    if tokens & {"flair", "t2f", "t2flair"} or "flair" in name:
        return "flair"
    if tokens & {"t1ce", "t1c", "t1gd", "t1post", "t1wce", "gd", "ce", "contrast"}:
        return "t1ce"
    if tokens & {"t2", "t2w"}:
        return "t2"
    if tokens & {"t1", "t1n", "t1w"}:
        return "t1"
    return "other"

# words in file names that describe the scan or its processing, not the timepoint
NOT_TIMEPOINT = {"seg", "segmentation", "segmentations", "mask", "tumormask", "label", "labels", "labelmap", "adc",
                 "flair", "t2f", "t2flair", "t1ce", "t1c", "t1gd", "t1post", "t1wce", "gd", "ce", "contrast", "t2", "t2w",
                 "t1", "t1n", "t1w", "brain", "skull", "stripped", "skullstripped", "n4", "z", "zscore", "mni152",
                 "sri24", "reg", "registered", "norm", "normalized", "nii", "gz"}

def subject_and_timepoint(cfg, path):
    dirs, fname = path.split("/")[:-1], path.split("/")[-1]
    m = re.search(cfg["subject"], path) if cfg["subject"] else None
    if m:
        subject = m.group(1)
    else:  # fallback: the closest folder with a 3+ digit number in its name, else the start of the file name
        found = [d for d in reversed(dirs) if re.search(r"\d{3,}", d) and d != cfg["folder"]]
        subject = found[0] if found else re.split(r"[_\-]", fname)[0]
    if cfg["single_timepoint"]:
        return subject, "0"
    m = re.search(cfg["timepoint"], path) if cfg["timepoint"] else None
    if m:
        return subject, m.group(1)
    i = next((i for i, d in enumerate(dirs) if subject in d), None)
    if i is not None and i + 1 < len(dirs):
        return subject, dirs[i + 1]         # a folder below the patient folder = the timepoint
    rest = re.sub(r"\.nii(\.gz)?$", "", fname).replace(subject, "")   # else: what is left of the file name
    left = [w for w in re.split(r"[_\-\.\s]+", rest.lower()) if w and w not in NOT_TIMEPOINT]
    return subject, "_".join(left) or "0"

def timepoint_order(label):
    """Sort key: pre-op < post-op < recurrence, then numbers in numeric order."""
    low = label.lower()
    stage = 0 if "pre" in low else 1 if "post" in low else 2 if ("recur" in low or "prog" in low) else 1
    nums = [int(n) for n in re.findall(r"\d+", label)]
    return (stage if not nums else 0, nums[-1] if nums else 0, low)

# quick look at the first files of each archive, to check the rules above
for key, cfg in present.items():
    archive = sorted(glob.glob(f"{RAW}/{cfg['folder']}/*.tar"), key=os.path.getsize)[-1]
    print(f"--- {key}: {os.path.basename(archive)}")
    with tarfile.open(archive, "r|") as t:
        shown = 0
        for m in t:
            if m.isfile() and m.name.endswith((".nii", ".nii.gz")):
                s, tp = subject_and_timepoint(cfg, m.name)
                print(f"   {m.name[:90]:90}  -> {s} | tp {tp} | {scan_type(os.path.basename(m.name))}")
                shown += 1
                if shown == 6:
                    break

In [ ]:
# 3. Read every archive: checksum, shape and voxel size of every scan; label values of every mask
def nifti_info(data, gz, is_seg):
    """Shape, voxel size and data type from the header; for masks also the label values (needs the full image)."""
    if is_seg:
        raw = gzip.decompress(data) if gz else data
    else:  # the header is in the first 348 bytes: no need to unpack the whole scan
        raw = gzip.GzipFile(fileobj=io.BytesIO(data)).read(1024) if gz else data[:1024]
    hdr = nib.Nifti1Header.from_fileobj(io.BytesIO(raw))
    shape = "x".join(str(s) for s in hdr.get_data_shape())
    spacing = "x".join(f"{z:g}" for z in hdr.get_zooms()[:3])
    labels = ""
    if is_seg:
        arr = np.asanyarray(nib.Nifti1Image.from_bytes(raw).dataobj)
        labels = ",".join(str(int(round(v))) for v in np.unique(arr))
    return shape, spacing, str(hdr.get_data_dtype()), labels

def read_sums(text):
    """Parse a checksum file: lines like '<hex>  <path>'."""
    out = {}
    for line in text.splitlines():
        m = re.match(r"\s*([0-9a-fA-F]{32,64})\s+\*?(.+?)\s*$", line)
        if m:
            out[os.path.basename(m.group(2))] = m.group(1).lower()
    return out

for key, cfg in present.items():
    target = f"{OUT}/{key}_manifest.csv"
    if os.path.exists(target) and not REBUILD:
        print(f"{key}: manifest already exists, skipping (set REBUILD = True to redo)")
        continue
    rows, sums = [], {}
    # BraTS 2021: the big archive first, then the single-patient ones (same order as extraction)
    archives = sorted(glob.glob(f"{RAW}/{cfg['folder']}/*.tar"), key=lambda p: -os.path.getsize(p))
    for archive in archives:
        start, done_bytes = datetime.datetime.now(), 0
        print(f"{key}: reading {os.path.basename(archive)} ({os.path.getsize(archive) / 1e9:.1f} GB) ...")
        with tarfile.open(archive, "r|") as t:
            for m in t:
                if not m.isfile():
                    continue
                data = t.extractfile(m).read()
                done_bytes += len(data)
                if m.name.endswith(".sums"):
                    sums.update(read_sums(data.decode("utf-8", "ignore")))
                    continue
                if not m.name.endswith((".nii", ".nii.gz")):
                    continue
                fname = os.path.basename(m.name)
                kind = scan_type(fname)
                subject, tp = subject_and_timepoint(cfg, m.name)
                try:
                    shape, spacing, dtype, labels = nifti_info(data, m.name.endswith(".gz"), kind == "seg")
                    error = ""
                except Exception as e:
                    shape = spacing = dtype = labels = ""
                    error = f"unreadable: {type(e).__name__}"
                rows.append(dict(
                    cohort=key, subject_id=subject, timepoint_raw=tp, scan=kind, file_name=fname,
                    path_in_archive=m.name, archive=os.path.basename(archive),
                    extracted_path=(f"{cfg['folder']}/{m.name}" if key == "brats2021" else m.name),
                    size_bytes=len(data), sha256=hashlib.sha256(data).hexdigest(), md5=hashlib.md5(data).hexdigest(),
                    image_shape=shape, spacing_mm=spacing, data_type=dtype, label_values=labels, error=error))
                if len(rows) % 1000 == 0:
                    secs = (datetime.datetime.now() - start).total_seconds()
                    print(f"   {len(rows)} files, {done_bytes / 1e9:.1f} GB, {done_bytes / 1e6 / max(secs, 1):.0f} MB/s")
    df = pd.DataFrame(rows)
    if sums:  # compare with the checksum file that came with the dataset
        def check(r):
            ref = sums.get(r.file_name)
            return "" if ref is None else "match" if ref in (r.md5, r.sha256) else "MISMATCH"
        df["source_checksum"] = df.apply(check, axis=1)
    # metadata files stored next to the archive (clinical tables etc.)
    for extra in sorted(glob.glob(f"{RAW}/{cfg['folder']}/*")):
        if not extra.endswith(".tar") and os.path.isfile(extra):
            blob = open(extra, "rb").read()
            df.loc[len(df)] = dict(cohort=key, scan="metadata", file_name=os.path.basename(extra),
                                   extracted_path=f"{cfg['folder']}/{os.path.basename(extra)}", size_bytes=len(blob),
                                   sha256=hashlib.sha256(blob).hexdigest(), md5=hashlib.md5(blob).hexdigest())
    df.to_csv(target, index=False)
    print(f"{key}: saved {os.path.basename(target)} ({len(df)} rows)\n")

In [ ]:
# 4. Build unified_subject_index.csv (one row per patient + timepoint) and check every dataset
TEMPLATE_COLUMNS = ("canonical_subject_id,dataset_source,dataset_version,cohort_role,original_subject_id,original_case_id,"
    "mapped_brats21_id,study_id,timepoint_index,timepoint_label,scan_date,days_from_baseline,scan_order_source,treatment_stage,"
    "T1_available,T1ce_available,T2_available,FLAIR_available,segmentation_available,segmentation_type,mask_refinement,"
    "original_label_scheme,canonical_label_scheme_applied,ET_present,RC_present,tumor_type,who_grade,idh_status,mgmt_status,"
    "mgmt_label_source,molecular_labels_available,clinical_labels_available,longitudinal_group,num_timepoints_for_subject,split,"
    "split_version,lockbox,overlap_checked_against,overlap_flag,overlap_evidence,institution_or_site,country,"
    "scanner_field_strength,voxel_spacing_raw_mm,volume_shape_raw,preprocessing_state_raw,file_format,raw_path,processed_path,"
    "sha256_t1,sha256_t1ce,sha256_t2,sha256_flair,sha256_seg,license,access_agreement_accepted_by,download_date,qc_status,"
    "qc_issues,excluded,exclusion_reason,notes").split(",")
CORE = ["t1", "t1ce", "t2", "flair"]
index_rows, report = [], []

for key, cfg in COHORTS.items():
    path = f"{OUT}/{key}_manifest.csv"
    if not os.path.exists(path):
        continue
    df = pd.read_csv(path, dtype=str, keep_default_na=False)
    scans = df[df.scan != "metadata"]
    scans = scans.drop_duplicates(["path_in_archive"], keep="last")  # same file in two archives: the later one wins (BraTS 00495/00621)
    download_date = datetime.date.fromtimestamp(
        os.path.getmtime(sorted(glob.glob(f"{RAW}/{cfg['folder']}/*.tar"))[0])).isoformat()
    n_tp = scans.groupby("subject_id").timepoint_raw.nunique()
    issues_seen = collections.Counter()
    for (subject, tp_raw), g in scans.groupby(["subject_id", "timepoint_raw"]):
        tps = sorted(scans[scans.subject_id == subject].timepoint_raw.unique(), key=timepoint_order)
        idx = tps.index(tp_raw)
        pick = {k: g[g.scan == k].sort_values("path_in_archive") for k in CORE + ["seg"]}
        first = {k: (v.iloc[0] if len(v) else None) for k, v in pick.items()}
        issues = [f"missing_{k}" for k in CORE if first[k] is None]
        if first["seg"] is None:
            issues.append("missing_seg")
        extra = [k for k, v in pick.items() if len(v) > 1]
        if extra:
            issues.append("several_files_for_" + "+".join(extra))
        shapes = {r["image_shape"] for r in first.values() if r is not None and r["image_shape"]}
        if len(shapes) > 1:
            issues.append("shape_differs_between_scans")
        if (g.error != "").any():
            issues.append("unreadable_file")
        if "source_checksum" in g and (g.source_checksum == "MISMATCH").any():
            issues.append("checksum_mismatch")
        issues_seen.update(issues)
        labels = first["seg"].label_values if first["seg"] is not None else ""
        label_set = {int(v) for v in labels.split(",") if v != ""}
        ref = first["t1ce"] if first["t1ce"] is not None else next((r for r in first.values() if r is not None), None)
        row = dict.fromkeys(TEMPLATE_COLUMNS, "")
        row.update(
            canonical_subject_id=f"{key}__{subject}", dataset_source=key, original_subject_id=subject,
            original_case_id=subject if n_tp[subject] == 1 else f"{subject}/{tp_raw}",
            mapped_brats21_id=subject if key == "brats2021" else "",
            timepoint_index=idx, timepoint_label=f"tp{idx:02d}", scan_order_source=cfg["order"],
            treatment_stage=cfg["stage"],
            T1_available=int(first["t1"] is not None), T1ce_available=int(first["t1ce"] is not None),
            T2_available=int(first["t2"] is not None), FLAIR_available=int(first["flair"] is not None),
            segmentation_available=int(first["seg"] is not None), segmentation_type=cfg["seg_type"],
            mask_refinement=cfg["refinement"], original_label_scheme=cfg["label_scheme"], canonical_label_scheme_applied=0,
            ET_present="" if cfg["et"] is None or not labels else int(cfg["et"] in label_set),
            RC_present="" if cfg["rc"] is None or not labels else int(cfg["rc"] in label_set),
            tumor_type=cfg["tumor"],
            longitudinal_group=subject if n_tp[subject] > 1 else "", num_timepoints_for_subject=int(n_tp[subject]),
            institution_or_site=cfg["site"], country=cfg["country"],
            voxel_spacing_raw_mm=ref["spacing_mm"] if ref is not None else "", volume_shape_raw=ref["image_shape"] if ref is not None else "",
            preprocessing_state_raw=cfg["prep"], file_format="nifti_gz",
            raw_path="data/raw/" + os.path.dirname(ref.extracted_path) if ref is not None else "",
            sha256_t1=first["t1"].sha256 if first["t1"] is not None else "",
            sha256_t1ce=first["t1ce"].sha256 if first["t1ce"] is not None else "",
            sha256_t2=first["t2"].sha256 if first["t2"] is not None else "",
            sha256_flair=first["flair"].sha256 if first["flair"] is not None else "",
            sha256_seg=first["seg"].sha256 if first["seg"] is not None else "",
            license=cfg["license"], download_date=download_date, qc_status="pending", qc_issues=";".join(issues),
            excluded=0, notes=f"mask labels: {labels}" if labels else "")
        index_rows.append(row)
    subjects, timepoints = scans.subject_id.nunique(), scans.groupby(["subject_id", "timepoint_raw"]).ngroups
    exp_s, exp_t = cfg["expect"]
    ok = subjects == exp_s and abs(timepoints - exp_t) <= 0.05 * exp_t
    report.append((key, subjects, exp_s, timepoints, exp_t, ok, issues_seen, scans))

index = pd.DataFrame(index_rows, columns=TEMPLATE_COLUMNS)
index.to_csv(f"{OUT}/unified_subject_index.csv", index=False)
print(f"Saved unified_subject_index.csv: {len(index)} rows\n")

for key, s, es, t, et, ok, issues, scans in report:
    print(f"===== {key}: {s} patients (expected {es}), {t} patient-timepoints (expected about {et})  {'OK' if ok else 'CHECK'}")
    print("  scans:", {k: int(v) for k, v in scans.scan.value_counts().items()})
    print("  mask label values:", dict(collections.Counter(scans[scans.scan == "seg"].label_values)))
    print("  shapes:", dict(collections.Counter(scans[scans.scan != "seg"]["image_shape"]).most_common(3)),
          "| voxel sizes:", dict(collections.Counter(scans.spacing_mm).most_common(3)))
    print("  issues:", dict(issues) or "none")
    if "source_checksum" in scans:
        print("  checksums vs TCIA file:", {k: int(v) for k, v in scans.source_checksum.replace("", "not listed").value_counts().items()})

In [ ]:
# 5. Finish writing to Drive
drive.flush_and_unmount()
print("Done. The CSVs are in MyDrive/BrainMRI_Data/manifests/.")

## What to do with the result

- **Everything says OK:** download the CSVs from `MyDrive/BrainMRI_Data/manifests/` into the repo's `data/manifests/`, commit them
  on the `M1-T03-patient-list` branch, open a pull request, and set M1-T03 to **In Review**.
- **A dataset says CHECK, or shows issues:** don't fix anything by hand. Send the step 2 and step 4 output to the person
  maintaining these notebooks; the file-name rules in step 2 may need adjusting for that dataset.
- **Two files per scan in UTSW** (`several_files_for_...`) is expected: it ships two skull-stripped versions. M1-T04 decides which one we keep.
- **Mask label values** answer an open question from `docs/dataset_research.md` (section 14): which numbers RHUH-GBM and UTSW use for each tumor region.